# Baseline pipeline (v2 — label-mapping fix)

Same TF-IDF + classical-model baseline as `baseline_pipeline_notebook.ipynb`, with the
label-mapping bug fixed (see `claude-workspace/ISSUE_PLAN.md`, issue I-1) and a corrected
evaluation: macro-F1 as the primary metric, per-class precision/recall/F1, confusion
matrices, and Dummy baselines (I-2, I-5). Labels and evaluation come from `liar_utils.py`
so the baseline and proposed pipelines can never diverge (I-4).

In [1]:
import re

import pandas as pd
from nltk.corpus import stopwords
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

from liar_utils import RANDOM_STATE, evaluate_full, load_and_label, print_report

Load data with the corrected label mapping

In [2]:
train_df = load_and_label("train.csv")
test_df = load_and_label("test.csv")
valid_df = load_and_label("valid.csv")

balance = train_df["Label"].value_counts(normalize=True).rename({0: "fake", 1: "real"})
print("Train class balance:\n", balance)
fake_share = balance["fake"]
assert 0.35 < fake_share < 0.5, "Fake-class share outside the expected ~44% range -- check label mapping"

Train class balance:
 Label
real    0.561719
fake    0.438281
Name: proportion, dtype: float64


Preprocess text

In [3]:
stop_words = set(stopwords.words("english"))


def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    tokens = text.split()
    tokens = [w for w in tokens if w not in stop_words]
    return " ".join(tokens)


train_df["clean_text"] = train_df["Statement"].apply(preprocess_text)
test_df["clean_text"] = test_df["Statement"].apply(preprocess_text)
valid_df["clean_text"] = valid_df["Statement"].apply(preprocess_text)

TF-IDF feature extraction

In [4]:
X_train = train_df["clean_text"]
X_test = test_df["clean_text"]
X_valid = valid_df["clean_text"]

y_train = train_df["Label"]
y_test = test_df["Label"]
y_valid = valid_df["Label"]

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)
X_valid_tfidf = vectorizer.transform(X_valid)

Dummy baselines (I-5) -- the bar every real model must clear

In [5]:
dummy_most_frequent = DummyClassifier(strategy="most_frequent")
dummy_stratified = DummyClassifier(strategy="stratified", random_state=RANDOM_STATE)

dummy_most_frequent.fit(X_train_tfidf, y_train)
dummy_stratified.fit(X_train_tfidf, y_train)

DummyClassifier(random_state=42, strategy='stratified')

Models -- including class_weight='balanced' variants of LR/SVM (I-5)

In [6]:
models = {
    "Dummy (most frequent)": dummy_most_frequent,
    "Dummy (stratified)": dummy_stratified,
    "Naive Bayes": MultinomialNB().fit(X_train_tfidf, y_train),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE).fit(
        X_train_tfidf, y_train
    ),
    "Logistic Regression (balanced)": LogisticRegression(
        max_iter=1000, random_state=RANDOM_STATE, class_weight="balanced"
    ).fit(X_train_tfidf, y_train),
    "SVM": LinearSVC(random_state=RANDOM_STATE).fit(X_train_tfidf, y_train),
    "SVM (balanced)": LinearSVC(random_state=RANDOM_STATE, class_weight="balanced").fit(
        X_train_tfidf, y_train
    ),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE).fit(
        X_train_tfidf, y_train
    ),
}

Evaluate -- macro-F1 primary, per-class P/R/F1, confusion matrix (I-2)

In [7]:
rows = []
for name, model in models.items():
    y_test_pred = model.predict(X_test_tfidf)
    y_valid_pred = model.predict(X_valid_tfidf)
    print_report(name, y_test, y_test_pred)

    test_metrics = evaluate_full(y_test, y_test_pred)
    valid_metrics = evaluate_full(y_valid, y_valid_pred)
    rows.append(
        {
            "Pipeline": "Baseline",
            "Method": "TF-IDF only",
            "Model": name,
            "Valid Accuracy": valid_metrics["accuracy"],
            "Valid Macro-F1": valid_metrics["macro_f1"],
            "Valid Fake F1": valid_metrics["fake_f1"],
            "Test Accuracy": test_metrics["accuracy"],
            "Test Macro-F1": test_metrics["macro_f1"],
            "Test Fake Precision": test_metrics["fake_precision"],
            "Test Fake Recall": test_metrics["fake_recall"],
            "Test Fake F1": test_metrics["fake_f1"],
            "Test Real F1": test_metrics["real_f1"],
            "Test Confusion Matrix": test_metrics["confusion_matrix"],
        }
    )

baseline_results = pd.DataFrame(rows)
baseline_results


Dummy (most frequent)
[[  0 553]
 [  0 714]]
              precision    recall  f1-score   support

        fake      0.000     0.000     0.000       553
        real      0.564     1.000     0.721       714

    accuracy                          0.564      1267
   macro avg      0.282     0.500     0.360      1267
weighted avg      0.318     0.564     0.406      1267


Dummy (stratified)
[[236 317]
 [326 388]]
              precision    recall  f1-score   support

        fake      0.420     0.427     0.423       553
        real      0.550     0.543     0.547       714

    accuracy                          0.493      1267
   macro avg      0.485     0.485     0.485      1267
weighted avg      0.493     0.493     0.493      1267


Naive Bayes
[[221 332]
 [163 551]]
              precision    recall  f1-score   support

        fake      0.576     0.400     0.472       553
        real      0.624     0.772     0.690       714

    accuracy                          0.609      1267
   

/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{m


Random Forest
[[229 324]
 [177 537]]
              precision    recall  f1-score   support

        fake      0.564     0.414     0.478       553
        real      0.624     0.752     0.682       714

    accuracy                          0.605      1267
   macro avg      0.594     0.583     0.580      1267
weighted avg      0.598     0.605     0.593      1267



,Pipeline,Method,Model,Valid Accuracy,Valid Macro-F1,Valid Fake F1,Test Accuracy,Test Macro-F1,Test Fake Precision,Test Fake Recall,Test Fake F1,Test Real F1,Test Confusion Matrix
0,Baseline,TF-IDF only,Dummy (most frequent),0.520249,0.342213,0.000000,0.563536,0.360424,0.000000,0.000000,0.000000,0.720848,"[[0, 553], [0, 714]]"
1,Baseline,TF-IDF only,Dummy (stratified),0.519470,0.516597,0.479325,0.492502,0.485091,0.419929,0.426763,0.423318,0.546864,"[[236, 317], [326, 388]]"
2,Baseline,TF-IDF only,Naive Bayes,0.611371,0.597663,0.523400,0.609313,0.580881,0.575521,0.399638,0.471718,0.690044,"[[221, 332], [163, 551]]"
3,Baseline,TF-IDF only,Logistic Regression,0.612928,0.603981,0.544455,0.621942,0.600735,0.587678,0.448463,0.508718,0.692752,"[[248, 305], [174, 540]]"
4,Baseline,TF-IDF only,Logistic Regression (balanced),0.605919,0.605642,0.595200,0.602210,0.597778,0.542169,0.569620,0.555556,0.640000,"[[315, 238], [266, 448]]"
5,Baseline,TF-IDF only,SVM,0.593458,0.591173,0.560606,0.602210,0.591301,0.548323,0.502712,0.524528,0.658073,"[[278, 275], [229, 485]]"
6,Baseline,TF-IDF only,SVM (balanced),0.591121,0.590851,0.580336,0.591160,0.586878,0.529915,0.560579,0.544815,0.628940,"[[310, 243], [275, 439]]"
7,Baseline,TF-IDF only,Random Forest,0.619938,0.609492,0.545624,0.604578,0.579743,0.564039,0.414105,0.477581,0.681905,"[[229, 324], [177, 537]]"


In [8]:
baseline_results.to_csv("baseline_results_v2.csv", index=False)